In [10]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import os
import base64
from datetime import datetime

# --- FILE SETUP ---
FILE = "bill_history.csv"
if os.path.exists(FILE):
    history_df = pd.read_csv(FILE)
else:
    history_df = pd.DataFrame(columns=["Date", "Bill", "Total", "Result"])

# --- CSS FIXES & UI STYLING ---
style = HTML("""
    <style>
        .widget-label { min-width: 160px !important; }
        
        /* Summary Box: Darker background for high contrast */
        .summary-box {
            background-color: #1e293b; 
            border-left: 5px solid #22c55e;
            border: 1px solid #334155;
            padding: 15px;
            margin-bottom: 20px;
            border-radius: 8px;
            color: #f8fafc !important; /* Forces white text */
            font-family: sans-serif;
        }
        
        /* Button spacing fix */
        .widget-hbox > .widget-subset, .widget-hbox > .jupyter-button {
            margin-right: 8px !important;
        }

        .download-btn {
            cursor:pointer; 
            background-color:#f8fafc; 
            border:1px solid #cbd5e1; 
            padding:6px 12px; 
            border-radius:6px; 
            font-size: 13px; 
            color: #0f172a;
            font-weight: bold;
        }
    </style>
""")

In [11]:
# Create Input Widgets
title_input = widgets.Text(description="Bill Name:", placeholder="e.g. Electric Bill")
andrew_input = widgets.FloatText(description="Andrew Paid ($):", value=0.0)
moi_input = widgets.FloatText(description="Moi Paid ($):", value=0.0)

# Buttons & Checkbox
calc_button = widgets.Button(description="Calculate & Save", button_style='success')
delete_button = widgets.Button(description="Delete Last", button_style='danger')
confirm_delete = widgets.Checkbox(value=False, description='Confirm', indent=False)

# Placeholders
summary_output = widgets.Output()
download_output = widgets.Output()
output_area = widgets.Output()

# --- THE FIX: Tightly packed button row ---
button_row = widgets.HBox(
    [calc_button, delete_button, confirm_delete, download_output],
    layout=widgets.Layout(display='flex', flex_flow='row', justify_content='flex-start', align_items='center')
)

layout = widgets.VBox([
    widgets.HTML("<h2 style='color: #f8fafc;'>⚖️ Andrew & Moi: Bill Balancer</h2>"),
    summary_output, 
    title_input, 
    andrew_input, 
    moi_input, 
    widgets.HTML("<br>"), # Spacing
    button_row, 
    output_area
])

In [12]:
def display_summary():
    with summary_output:
        clear_output()
        if not history_df.empty:
            current_month = datetime.now().strftime("%Y-%m")
            temp_df = history_df.copy()
            temp_df['Total_Num'] = temp_df['Total'].replace(r'[\$,]', '', regex=True).astype(float)
            monthly_data = temp_df[temp_df['Date'].str.startswith(current_month)]
            monthly_total = monthly_data['Total_Num'].sum()
            display(HTML(f'<div class="summary-box"><strong>📊 {datetime.now().strftime("%B %Y")} Summary</strong><br>Total Spent: ${monthly_total:.2f} across {len(monthly_data)} entries.</div>'))

def create_download_link():
    if os.path.exists(FILE):
        df = pd.read_csv(FILE)
        formatted_df = df.rename(columns={"Date": "Date Settled", "Bill": "Description", "Total": "Total Amount", "Result": "Settlement Status"})
        csv_data = formatted_df.to_csv(index=False)
        b64 = base64.b64encode(csv_data.encode()).decode()
        filename = f"Bill_Balancer_{datetime.now().strftime('%Y_%m_%d')}.csv"
        return HTML(f'<a href="data:file/csv;base64,{b64}" download="{filename}" style="text-decoration:none;"><button class="download-btn">📥 Download for Excel</button></a>')
    return HTML("")

def update_history_table():
    with output_area:
        clear_output()
        if not history_df.empty:
            display(HTML("<h3>📝 Recent History</h3>"))
            display(history_df.tail(10)[::-1])

def on_click(b):
    global history_df
    total = andrew_input.value + moi_input.value
    share = total / 2
    res = f"Moi owes Andrew ${andrew_input.value-share:.2f}" if andrew_input.value > share else f"Andrew owes Moi ${moi_input.value-share:.2f}" if moi_input.value > share else "Even split!"
    new_row = {"Date": datetime.now().strftime("%Y-%m-%d"), "Bill": title_input.value, "Total": f"${total:.2f}", "Result": res}
    history_df = pd.concat([history_df, pd.DataFrame([new_row])], ignore_index=True)
    history_df.to_csv(FILE, index=False)
    refresh_ui()

def on_delete(b):
    global history_df
    # SAFETY CHECK: Only delete if the checkbox is checked
    if confirm_delete.value and not history_df.empty:
        history_df = history_df.drop(history_df.index[-1])
        history_df.to_csv(FILE, index=False)
        confirm_delete.value = False # Reset the checkbox after deletion
        refresh_ui()
    elif not confirm_delete.value:
        with output_area:
            print("⚠️ Please check 'Confirm Delete' first!")

def refresh_ui():
    display_summary()
    update_history_table()
    with download_output:
        clear_output(); display(create_download_link())

# Bind and Show
calc_button.on_click(on_click)
delete_button.on_click(on_delete)
display(style, layout)
refresh_ui()